# ChannelNet (Soltani et al. 2019) - full reproduction on Kaggle

**Before running**
1. Upload `ChannelNet_kaggle.zip` (code + Vienna channel data `data/vienna_*.mat`) as a Kaggle *Dataset* (Datasets -> New Dataset -> upload the zip). Kaggle extracts it automatically.
2. In this notebook: **Add Input** -> select that dataset.
3. Settings (right panel): **Accelerator = GPU T4 x2** (recommended: tensor cores make the fp16 training fast; P100 also works but is slower), **Internet = On** (only needed for the PDF report packages).
4. Use **Save Version -> Save & Run All (Commit)**. The notebook then runs in the background (up to 12 h) and everything in `/kaggle/working` is kept as output.

The default configuration trains 11 ChannelNet models (paper Figs. 3-5: VehA/SUI5 at 12 and 22 dB, VehA at 48/36/24/16/8 pilots at 20 dB, plus multi-SNR and SUI5 pilot extras) on 40 000 channel realisations each. Expected time on a P100/T4: about 3-5 h. Every step skips work that is already done, so an interrupted run can simply be resumed.

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 1. Copy the code into the writable working directory

In [ ]:
import glob, os, shutil
cands = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/run_all.py', recursive=True)]
print('found code at:', cands)
assert cands, 'Add the uploaded ChannelNet dataset as an input of this notebook!'
WORK = '/kaggle/working/ChannelNet'
shutil.copytree(cands[0], WORK, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('smoke_run', '__pycache__', '*.npy'))
os.chdir(WORK)
print(os.listdir('.'))

## 2. Dependencies (numpy / scipy / torch / matplotlib / pyyaml are preinstalled on Kaggle)

In [ ]:
!pip install -q markdown xhtml2pdf || echo "PDF packages not installed (Internet off?) - report.html is still produced"

## 3. Sanity checks of the OFDM / channel / estimator building blocks

In [ ]:
!python tests/test_sanity.py

## 4. Full pipeline
`data -> viz -> conventional -> train -> test -> plots -> report`

The ground-truth channels are the Vienna 5G simulator files in `data/vienna_*.mat` (generated with MATLAB, see `matlab/`). The intermediate cache goes to `/tmp` so that it doesn't fill the saved output.

In [ ]:
DATA = '/tmp/channelnet_data'
!python run_all.py --config configs/default.yaml --set paths.data_dir={DATA}

### (optional) split the training over several sessions
If you want shorter sessions, run only a subset per session, e.g.
```
!python scripts/04_train.py --set paths.data_dir=/tmp/channelnet_data --channel VehA --pilots 48
```
Then download the `checkpoints/` folder, re-upload it, and run the remaining steps; existing checkpoints are skipped.

## 5. Inspect results

In [ ]:
import pandas as pd
from IPython.display import Image, display
for f in ['metrics/mse_vs_snr_VehA_p48.csv', 'metrics/mse_vs_snr_SUI5_p48.csv',
          'metrics/mse_vs_pilots_VehA_snr22.csv', 'metrics/mse_vs_pilots_SUI5_snr22.csv']:
    p = os.path.join('results', f)
    if os.path.exists(p):
        print(f); display(pd.read_csv(p).style.format('{:.3e}'))
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(filename=f, width=900))

## 6. Package everything for download (Output tab -> `channelnet_outputs.zip`)

In [ ]:
os.chdir('/kaggle/working')
shutil.make_archive('/kaggle/working/channelnet_outputs', 'zip', root_dir='/kaggle/working',
                    base_dir='ChannelNet')
print(os.path.getsize('/kaggle/working/channelnet_outputs.zip') / 1e6, 'MB')